# Data adapter — NCHS birth cohorts

Framework stage: data adapter (S2). Reads the *Period/Cohort Linked Birth/Infant Death* files directly from the ZIP files in `../NCHS_data/raw/` and builds each birth cohort 2016–2023 as the user guide describes: births of year N joined with the deaths in years N and N+1 of infants born in N, by cohort sequence number and year of death. Writes one Parquet per cohort in `data/interim/` (not versioned) with the original codes and the labels. Spanish version: `01_build_cohorts_es.ipynb` (same code).

Requirements: `pandas`, `numpy` and `pyarrow` (`%pip install pyarrow` if missing). Each cohort takes about 2 minutes and about 3 GB of memory.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from libraries.nchs_reader import build_cohort, COHORTS, OUT_DIR

REBUILD = False      # True: rebuild every cohort even if its Parquet file exists
OUT_FILE = ROOT / "data" / "outputs_01_build_cohorts.txt"

## 1. Build the cohorts (existing ones are skipped)

In [ ]:
summaries = []
for y in COHORTS:
    if REBUILD or not (OUT_DIR / f"cohort_{y}.parquet").exists():
        summaries.append(build_cohort(y))          # about 2 minutes and 3 GB of memory per cohort
    else:
        print(f"cohort {y}: already built")

## 2. Aggregate counts per cohort

Population: live births to US-resident mothers, 22 to 47 weeks. Output: `data/outputs_01_build_cohorts.txt`.

In [ ]:
rows = []
for y in COHORTS:
    d = pd.read_parquet(OUT_DIR / f"cohort_{y}.parquet",
                        columns=["RESTATUS", "COMBGEST", "CO_SEQNUM", "died_infant", "label_main",
                                 "label_day1_6", "died_first_24h", "label_neonatal"])
    us = d["RESTATUS"] < 4
    ga = d["COMBGEST"]
    pop = us & ga.between(22, 47).fillna(False)
    p28 = us & ga.between(28, 47).fillna(False)
    rows.append({
        "cohort": y, "births": len(d), "births_us_resident": int(us.sum()),
        "infant_deaths_linked": int(d["died_infant"].sum()),
        "ga_unknown": int((us & (ga == 99)).sum()), "ga_lt22": int((us & (ga < 22)).sum()),
        "pop_ge22wk": int(pop.sum()),
        "events_main_pop": int(d.loc[pop, "label_main"].sum()),
        "events_first24h_pop": int(d.loc[pop, "died_first_24h"].sum()),
        "events_day1_6_pop": int(d.loc[pop, "label_day1_6"].sum()),
        "events_neonatal_pop": int(d.loc[pop, "label_neonatal"].sum()),
        "pop_ge28wk": int(p28.sum()), "events_main_ge28": int(d.loc[p28, "label_main"].sum()),
    })
t = pd.DataFrame(rows).set_index("cohort").T
lines = ["Data adapter (early neonatal component): birth cohorts built from the NCHS Period/Cohort Linked Birth/Infant Death "
         "public-use files (aggregate counts only)",
         "pop = live births to US-resident mothers with combined gestation 22-47 weeks; "
         "main label = death at 0-6 days", ""] + t.to_string().splitlines()
OUT_FILE.write_text("\n".join(lines) + "\n", encoding="utf-8")
t